Proyecto Olist ML
Predicción de retrasos logísticos y segmentación de clientes

Institución: Universidad de Cundinamarca

Programa: Ingeniería de Sistemas

Asignatura: Ciencia de Datos

Integrantes: Juan David Aguilar Páez y Nicolas Mauricio Vasquez

Cuadernillo: 03 - preparacion de datos

# Preparación de datos para Machine Learning — Olist


## Objetivo general

Preparar un conjunto de datos confiable para estudiar la predicción de retrasos de entrega en Olist, evitando utilizar información que todavía no se conocería al momento de predecir.

## Objetivos específicos

1. Integrar las tablas necesarias y trabajar con una observación por pedido.
2. Construir la variable objetivo, las características iniciales y una división temporal que respete la disponibilidad de las etiquetas.
3. Explorar dos características adicionales: días prometidos y distancia geográfica aproximada entre vendedores y clientes.
4. Conservar versiones independientes de los datos para comparar los modelos en el cuadernillo 04.

**Organización:** primero se prepara la base original (**V1**), después se investiga el plazo prometido (**V2**) y finalmente se construye la distancia logística (**V3**). Los resultados de evaluación de modelos pertenecen al cuadernillo 04.


## Parte I. Preparación inicial (V1)

### 1. Carga de archivos y funciones

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

sys.path.insert(0, str(Path.cwd().parent))
from src.preparacion import preparar_objetivo, preparar_articulos, preparar_envios

ruta = Path("../data/raw")

pedidos = pd.read_csv(ruta / "olist_orders_dataset.csv")
clientes = pd.read_csv(ruta / "olist_customers_dataset.csv")
items = pd.read_csv(ruta / "olist_order_items_dataset.csv")
productos = pd.read_csv(ruta / "olist_products_dataset.csv")
vendedores = pd.read_csv(ruta / "olist_sellers_dataset.csv")

pedidos.shape

(99441, 8)

### 2. Variable objetivo e integración por pedido

In [2]:
datos = preparar_objetivo(pedidos)
datos["retraso"].value_counts()

retraso
0    89936
1     6534
Name: count, dtype: int64

In [3]:
datos = datos.merge(
    clientes[["customer_id", "customer_state"]],
    on="customer_id", how="left", validate="many_to_one"
)

articulos = preparar_articulos(items, productos)
datos = datos.merge(articulos, on="order_id", how="left", validate="one_to_one")

envios = preparar_envios(items, vendedores, pedidos, clientes)
datos = datos.merge(envios, on="order_id", how="left", validate="one_to_one")

assert datos["order_id"].is_unique
assert len(datos) == 96470

datos.shape

(96470, 14)

In [4]:
datos[["customer_state", "cantidad_items", "peso_total_g",
       "costo_envio", "envio_entre_estados"]].isna().sum()

customer_state          0
cantidad_items          0
peso_total_g           16
costo_envio             0
envio_entre_estados     0
dtype: int64

### 3. Variables temporales y selección de características

In [5]:
datos["order_purchase_timestamp"] = pd.to_datetime(datos["order_purchase_timestamp"])
datos["mes_compra"] = datos["order_purchase_timestamp"].dt.month
datos["dia_semana"] = datos["order_purchase_timestamp"].dt.dayofweek
datos["hora_compra"] = datos["order_purchase_timestamp"].dt.hour

variables = [
    "customer_state", "cantidad_items", "peso_total_g", "costo_envio",
    "envio_entre_estados", "mes_compra", "dia_semana", "hora_compra"
]

base_modelo = datos[
    ["order_id", "order_purchase_timestamp"] + variables + ["retraso"]
].copy()

base_modelo.shape

C:\Users\juand\AppData\Local\Temp\ipykernel_31004\240428910.py:3: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  datos["dia_semana"] = datos["order_purchase_timestamp"].dt.dayofweek


(96470, 11)

In [6]:
base_modelo.isna().sum()

order_id                     0
order_purchase_timestamp     0
customer_state               0
cantidad_items               0
peso_total_g                16
costo_envio                  0
envio_entre_estados          0
mes_compra                   0
dia_semana                   0
hora_compra                  0
retraso                      0
dtype: int64

### 4. Separación temporal y disponibilidad de etiquetas

In [7]:
base_modelo = base_modelo.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

corte = int(len(base_modelo) * 0.8)
train = base_modelo.iloc[:corte].copy()
test = base_modelo.iloc[corte:].copy()

inicio_prueba = test["order_purchase_timestamp"].min()

pd.DataFrame({
    "Conjunto": ["Entrenamiento inicial", "Prueba"],
    "Pedidos": [len(train), len(test)],
    "Inicio": [train["order_purchase_timestamp"].min(),
               test["order_purchase_timestamp"].min()],
    "Fin": [train["order_purchase_timestamp"].max(),
            test["order_purchase_timestamp"].max()],
    "Tasa_retraso": [train["retraso"].mean(), test["retraso"].mean()]
})

,Conjunto,Pedidos,Inicio,Fin,Tasa_retraso
0,Entrenamiento inicial,77176,2016-09-15 12:16:38,2018-05-26 18:16:57,0.075930
1,Prueba,19294,2018-05-26 18:18:03,2018-08-29 15:00:37,0.034933


La etiqueta de un pedido solo puede conocerse una vez registrada su entrega. Para simular el entrenamiento al comienzo del período de prueba, se excluyen del entrenamiento los pedidos entregados en ese instante o después. El conjunto de prueba no se modifica.

In [8]:
train_fechas = train[["order_id"]].merge(
    datos[["order_id", "order_delivered_customer_date"]],
    on="order_id", validate="one_to_one"
)

etiquetas_futuras = (
    train_fechas["order_delivered_customer_date"] >= inicio_prueba
)

print("Pedidos con etiqueta futura:", etiquetas_futuras.sum())

pedidos_conocidos = train_fechas.loc[~etiquetas_futuras, "order_id"]
train = train[train["order_id"].isin(pedidos_conocidos)].copy()

assert train["order_id"].is_unique
assert test["order_id"].is_unique
assert not set(train["order_id"]).intersection(test["order_id"])

train.shape, test.shape

Pedidos con etiqueta futura: 2048


((75128, 11), (19294, 11))

In [9]:
pd.DataFrame({
    "Conjunto": ["Entrenamiento corregido", "Prueba"],
    "Pedidos": [len(train), len(test)],
    "Tasa_retraso": [train["retraso"].mean(), test["retraso"].mean()]
})

,Conjunto,Pedidos,Tasa_retraso
0,Entrenamiento corregido,75128,0.071065
1,Prueba,19294,0.034933


### 5. Variables de entrenamiento y prueba

In [10]:
X_train = train[variables]
y_train = train["retraso"]
X_test = test[variables]
y_test = test["retraso"]

X_train.shape, X_test.shape

((75128, 8), (19294, 8))

In [11]:
pd.DataFrame({
    "Entrenamiento": y_train.value_counts(),
    "Prueba": y_test.value_counts()
})

,Entrenamiento,Prueba
retraso,,
0,69789,18620
1,5339,674


### 6. Definición del preprocesamiento inicial

In [12]:
preprocesamiento = ColumnTransformer(
    transformers=[
        ("peso", SimpleImputer(strategy="median"), ["peso_total_g"]),
        ("estado", OneHotEncoder(handle_unknown="ignore"), ["customer_state"])
    ],
    remainder="passthrough"
)

preprocesamiento

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('peso', ...), ('estado', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

El preprocesamiento **solo se define** en este cuadernillo: no se ejecuta `fit` sobre los conjuntos. En el cuadernillo de modelado se incorporará a un `Pipeline` para que la imputación y la codificación se aprendan únicamente con los registros de entrenamiento de cada ajuste o partición de validación temporal.

### 7. Exportación de los conjuntos originales

In [13]:
salida = Path("../data/processed")
salida.mkdir(parents=True, exist_ok=True)

train.to_csv(salida / "train.csv", index=False)
test.to_csv(salida / "test.csv", index=False)

pd.DataFrame({
    "Archivo": ["train.csv", "test.csv"],
    "Registros": [len(pd.read_csv(salida / "train.csv")),
                  len(pd.read_csv(salida / "test.csv"))]
})

,Archivo,Registros
0,train.csv,75128
1,test.csv,19294


## Parte II. Ingeniería de características: plazo prometido (V2)

El plazo prometido se calcula como la diferencia entre la fecha estimada de entrega y la fecha de compra. Su uso como predictor al momento de compra **requiere verificar que la estimación estuviera disponible en ese momento**. No se utiliza la fecha real de entrega como característica predictora.

### 8. Construcción y exploración de `dias_prometidos`

In [14]:
datos["dias_prometidos"] = (
    datos["order_estimated_delivery_date"].dt.normalize()
    - datos["order_purchase_timestamp"].dt.normalize()
).dt.days

datos["dias_prometidos"].describe()

count    96470.000000
mean        24.372738
std          8.758428
min          3.000000
25%         19.000000
50%         24.000000
75%         29.000000
max        156.000000
Name: dias_prometidos, dtype: float64

In [15]:
datos.shape

(96470, 18)

### 9. Plazos prometidos y frecuencia de retrasos

In [16]:
datos["grupo_plazo"] = pd.cut(
    datos["dias_prometidos"],
    bins=[0, 10, 20, 30, 45, float("inf")],
    labels=["1-10", "11-20", "21-30", "31-45", "Más de 45"]
)

resumen_plazos = datos.groupby(
    "grupo_plazo",
    observed=True
)["retraso"].agg(
    pedidos="count",
    retrasos="sum",
    tasa_retraso="mean"
)

resumen_plazos

,pedidos,retrasos,tasa_retraso
grupo_plazo,,,
1-10,3945,413,0.104689
11-20,26719,1599,0.059845
21-30,45934,3537,0.077002
31-45,18004,952,0.052877
Más de 45,1868,33,0.017666


In [17]:
resumen_mensual = (
    datos.groupby(
        datos["order_purchase_timestamp"].dt.to_period("M")
    )
    .agg(
        pedidos=("retraso", "size"),
        dias_prometidos=("dias_prometidos", "mean"),
        tasa_retraso=("retraso", "mean")
    )
)

resumen_mensual.tail(12).round(4)

,pedidos,dias_prometidos,tasa_retraso
order_purchase_timestamp,,,
2017-09,4150,23.0708,0.0439
2017-10,4478,23.6565,0.0418
2017-11,7288,23.1966,0.1240
2017-12,5513,28.3142,0.0746
2018-01,7069,26.9402,0.0570
2018-02,6555,25.1641,0.1413
2018-03,7003,22.6653,0.1896
2018-04,6798,24.3145,0.0450
2018-05,6749,23.5275,0.0656


### 10. Incorporación del plazo a la base histórica de entrenamiento

In [18]:
train_v2 = train.merge(
    datos[["order_id", "dias_prometidos"]],
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Dimensiones:", train_v2.shape)
print("Valores faltantes:", train_v2["dias_prometidos"].isna().sum())

Dimensiones: (75128, 12)
Valores faltantes: 0


In [19]:
train_v2.to_csv(
    salida / "train_v2.csv",
    index=False
)

print("Archivo guardado:", (salida / "train_v2.csv").exists())
print("Dimensiones:", train_v2.shape)

Archivo guardado: True
Dimensiones: (75128, 12)


## Parte III. Ingeniería de características: distancia logística

Las coordenadas se aproximan a partir del prefijo postal. La distancia de Haversine mide separación geográfica en línea recta; **no** representa la ruta real del transporte.

### 11. Exploración del archivo geográfico

In [20]:
geolocalizacion = pd.read_csv(
    ruta / "olist_geolocation_dataset.csv"
)

print("Dimensiones:", geolocalizacion.shape)
print("Columnas:", geolocalizacion.columns.tolist())

geolocalizacion.head()

Dimensiones: (1000163, 5)
Columnas: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [21]:
print("Registros geográficos:", len(geolocalizacion))

print(
    "Códigos postales únicos:",
    geolocalizacion["geolocation_zip_code_prefix"].nunique()
)

print(
    "Coordenadas faltantes:",
    geolocalizacion[
        ["geolocation_lat", "geolocation_lng"]
    ].isna().sum().sum()
)


Registros geográficos: 1000163
Códigos postales únicos: 19015
Coordenadas faltantes: 0


### 12. Una ubicación representativa por prefijo postal

In [22]:
ubicaciones = (
    geolocalizacion
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitud=("geolocation_lat", "median"),
        longitud=("geolocation_lng", "median")
    )
    .reset_index()
)

ubicaciones.head()

,geolocation_zip_code_prefix,latitud,longitud
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313
3,1004,-23.549535,-46.634771
4,1005,-23.549612,-46.636532


In [23]:
print("Ubicaciones agrupadas:", ubicaciones.shape)
print(
    "Códigos postales duplicados:",
    ubicaciones["geolocation_zip_code_prefix"].duplicated().sum()
)

ubicaciones.head()

Ubicaciones agrupadas: (19015, 3)
Códigos postales duplicados: 0


,geolocation_zip_code_prefix,latitud,longitud
0,1001,-23.550381,-46.634027
1,1002,-23.548551,-46.635072
2,1003,-23.548977,-46.635313
3,1004,-23.549535,-46.634771
4,1005,-23.549612,-46.636532


### 13. Cobertura de clientes y vendedores

In [24]:
clientes_geo = pd.read_csv(
    ruta / "olist_customers_dataset.csv"
)

vendedores_geo = pd.read_csv(
    ruta / "olist_sellers_dataset.csv"
)

print("Clientes:", clientes_geo.shape)
print("Vendedores:", vendedores_geo.shape)

Clientes: (99441, 5)
Vendedores: (3095, 4)


In [25]:
codigos_geo = set(
    ubicaciones["geolocation_zip_code_prefix"]
)

cobertura_clientes = (
    clientes_geo["customer_zip_code_prefix"]
    .isin(codigos_geo)
    .mean() * 100
)

cobertura_vendedores = (
    vendedores_geo["seller_zip_code_prefix"]
    .isin(codigos_geo)
    .mean() * 100
)

print(f"Cobertura de clientes: {cobertura_clientes:.2f}%")
print(f"Cobertura de vendedores: {cobertura_vendedores:.2f}%")

Cobertura de clientes: 99.72%
Cobertura de vendedores: 99.77%


### 14. Relación entre artículos, vendedores y clientes

In [26]:
articulos_geo = pd.read_csv(
    ruta / "olist_order_items_dataset.csv"
)

articulos_vendedores = articulos_geo.merge(
    vendedores_geo[["seller_id", "seller_zip_code_prefix"]],
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("Artículos originales:", len(articulos_geo))
print("Artículos relacionados:", len(articulos_vendedores))

print(
    "Códigos postales faltantes:",
    articulos_vendedores["seller_zip_code_prefix"].isna().sum()
)

print(
    "Pedidos con varios vendedores:",
    (articulos_vendedores.groupby("order_id")["seller_id"].nunique() > 1).sum()
)

Artículos originales: 112650
Artículos relacionados: 112650
Códigos postales faltantes: 0
Pedidos con varios vendedores: 1278


In [27]:
articulos_vendedores = articulos_vendedores.merge(
    ubicaciones.rename(columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "latitud": "lat_vendedor",
        "longitud": "lon_vendedor"
    }),
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

print("Artículos:", len(articulos_vendedores))

print(
    "Artículos sin coordenadas:",
    articulos_vendedores["lat_vendedor"].isna().sum()
)

articulos_vendedores[
    ["order_id", "seller_id", "lat_vendedor", "lon_vendedor"]
].head()

Artículos: 112650
Artículos sin coordenadas: 253


,order_id,seller_id,lat_vendedor,lon_vendedor
0,00010242fe8c5a6d1ba2dd792cb16214,48436dade18ac8b2bce089ec2a041202,-22.498419,-44.125272
1,00018f77f2f0320c557190d7a144bdd3,dd7ddc04e1b6c2c614352b383efe2d36,-23.564289,-46.519045
2,000229ec398224ef6ca0657da4fc703e,5b51032eddd242adc84c38acab88f23d,-22.271648,-46.165556
3,00024acbcdf0a6daa1e931b038114c75,9d7a1d34a5052409006425275ba1c2b4,-20.554951,-47.387691
4,00042b26cf59d7ce69dfabb4e55b4fd9,df560393f3a51e74553ab94004ba5c87,-22.930408,-53.136438


In [28]:
clientes_ubicacion = clientes_geo[
    ["customer_id", "customer_zip_code_prefix"]
].merge(
    ubicaciones.rename(columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "latitud": "lat_cliente",
        "longitud": "lon_cliente"
    }),
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

print("Clientes:", len(clientes_ubicacion))

print(
    "Clientes sin coordenadas:",
    clientes_ubicacion["lat_cliente"].isna().sum()
)

clientes_ubicacion.head()

Clientes: 99441
Clientes sin coordenadas: 278


,customer_id,customer_zip_code_prefix,lat_cliente,lon_cliente
0,06b8999e2fba1a1fbc88172c00ba8bc7,14409,-20.502070,-47.396822
1,18955e83d337fd6b2def6b18a428ac77,9790,-23.727299,-46.542631
2,4e7b3e00288586ebd08712fdd0374a03,1151,-23.531294,-46.656404
3,b2b6027bc5c5109e529d4dc6358b12c3,8775,-23.497390,-46.182342
4,4f2d8ab171c80ec8364f7c12e35b23ad,13056,-22.973309,-47.141530


In [29]:
pedidos_geo = pd.read_csv(
    ruta / "olist_orders_dataset.csv"
)

pedidos_clientes = pedidos_geo[
    ["order_id", "customer_id"]
].merge(
    clientes_ubicacion[
        ["customer_id", "lat_cliente", "lon_cliente"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Pedidos originales:", len(pedidos_geo))
print("Pedidos relacionados:", len(pedidos_clientes))

print(
    "Pedidos sin coordenadas del cliente:",
    pedidos_clientes["lat_cliente"].isna().sum()
)

pedidos_clientes.head()

Pedidos originales: 99441
Pedidos relacionados: 99441
Pedidos sin coordenadas del cliente: 278


,order_id,customer_id,lat_cliente,lon_cliente
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,-23.576170,-46.587276
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,-12.126651,-45.008162
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,-16.744472,-48.514624
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,-5.774611,-35.273916
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,-23.675316,-46.515116


In [30]:
envios_geo = articulos_vendedores.merge(
    pedidos_clientes[
        ["order_id", "lat_cliente", "lon_cliente"]
    ],
    on="order_id",
    how="left",
    validate="many_to_one"
)

print("Artículos originales:", len(articulos_vendedores))
print("Artículos relacionados:", len(envios_geo))

print(
    "Artículos sin coordenadas completas:",
    envios_geo[
        ["lat_vendedor", "lon_vendedor",
         "lat_cliente", "lon_cliente"]
    ].isna().any(axis=1).sum()
)

envios_geo[
    ["order_id", "lat_vendedor", "lon_vendedor",
     "lat_cliente", "lon_cliente"]
].head()

Artículos originales: 112650
Artículos relacionados: 112650
Artículos sin coordenadas completas: 554


,order_id,lat_vendedor,lon_vendedor,lat_cliente,lon_cliente
0,00010242fe8c5a6d1ba2dd792cb16214,-22.498419,-44.125272,-21.762829,-41.310605
1,00018f77f2f0320c557190d7a144bdd3,-23.564289,-46.519045,-20.205737,-50.926924
2,000229ec398224ef6ca0657da4fc703e,-22.271648,-46.165556,-19.870383,-44.594355
3,00024acbcdf0a6daa1e931b038114c75,-20.554951,-47.387691,-23.104346,-46.595589
4,00042b26cf59d7ce69dfabb4e55b4fd9,-22.930408,-53.136438,-23.245512,-46.825182


### 15. Cálculo de distancias mediante Haversine

In [31]:
import numpy as np

lat1 = np.radians(envios_geo["lat_vendedor"])
lon1 = np.radians(envios_geo["lon_vendedor"])
lat2 = np.radians(envios_geo["lat_cliente"])
lon2 = np.radians(envios_geo["lon_cliente"])

a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin((lon2 - lon1) / 2) ** 2
)

envios_geo["distancia_km"] = (
    2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

print(envios_geo["distancia_km"].describe().round(2))
print(
    "Distancias faltantes:",
    envios_geo["distancia_km"].isna().sum()
)

count    112096.00
mean        596.88
std         590.33
min           0.00
25%         183.10
50%         431.72
75%         791.84
max        8677.86
Name: distancia_km, dtype: float64
Distancias faltantes: 554


### 16. Revisión de valores extremos

In [32]:
distancias_extremas = envios_geo[
    envios_geo["distancia_km"] > 4000
].copy()

print(
    "Artículos con distancia superior a 4.000 km:",
    len(distancias_extremas)
)

print(
    "Pedidos afectados:",
    distancias_extremas["order_id"].nunique()
)

distancias_extremas[
    [
        "order_id",
        "lat_vendedor",
        "lon_vendedor",
        "lat_cliente",
        "lon_cliente",
        "distancia_km"
    ]
].sort_values(
    "distancia_km",
    ascending=False
).head(10)

Artículos con distancia superior a 4.000 km: 4
Pedidos afectados: 4


,order_id,lat_vendedor,lon_vendedor,lat_cliente,lon_cliente,distancia_km
60915,8ad3f1d0f96992e43566c4c82c9f6c58,-26.231306,-49.412569,42.184003,-8.723762,8677.859564
34191,4d5abe7999d76d1fb6237d3677706af0,-21.258652,-47.813946,41.146203,-8.577855,8032.929717
75917,acdbc7396e191931c263db11af241d62,-21.258652,-47.813946,41.146203,-8.577855,8032.929717
60328,897ec6416d50126a9061626f0fc2d658,-23.652443,-46.755492,13.001420,-23.583939,4791.606339


In [33]:
revision_extremos = distancias_extremas[
    ["order_id", "lat_cliente", "lon_cliente", "distancia_km"]
].merge(
    pedidos_clientes[["order_id", "customer_id"]],
    on="order_id",
    validate="many_to_one"
).merge(
    clientes_geo[
        ["customer_id", "customer_zip_code_prefix", "customer_state"]
    ],
    on="customer_id",
    validate="many_to_one"
)

revision_extremos[
    [
        "order_id",
        "customer_zip_code_prefix",
        "customer_state",
        "lat_cliente",
        "lon_cliente",
        "distancia_km"
    ]
]

,order_id,customer_zip_code_prefix,customer_state,lat_cliente,lon_cliente,distancia_km
0,4d5abe7999d76d1fb6237d3677706af0,68275,PA,41.146203,-8.577855,8032.929717
1,897ec6416d50126a9061626f0fc2d658,46560,BA,13.001420,-23.583939,4791.606339
2,8ad3f1d0f96992e43566c4c82c9f6c58,83252,PR,42.184003,-8.723762,8677.859564
3,acdbc7396e191931c263db11af241d62,68275,PA,41.146203,-8.577855,8032.929717


### 17. Identificación de coordenadas sospechosas

In [34]:
ubicaciones_invalidas = ubicaciones[
    ~(
        ubicaciones["latitud"].between(-34, 6)
        & ubicaciones["longitud"].between(-74, -28)
    )
].copy()

print(
    "Prefijos postales con coordenadas sospechosas:",
    len(ubicaciones_invalidas)
)

ubicaciones_invalidas.head(10)

Prefijos postales con coordenadas sospechosas: 7


,geolocation_zip_code_prefix,latitud,longitud
6077,18243,28.008978,-15.536867
10691,46560,13.001420,-23.583939
13734,68275,41.146203,-8.577855
13748,68379,9.049063,-25.170903
15781,78131,38.816816,-9.394625
16419,83252,42.184003,-8.723762
18309,95130,14.585073,121.105394


### 18. Tratamiento de distancias fuera del rango geográfico

In [35]:
# Identificar coordenadas fuera del rango aproximado de Brasil
vendedor_valido = (
    envios_geo["lat_vendedor"].between(-34, 6)
    & envios_geo["lon_vendedor"].between(-74, -28)
)

cliente_valido = (
    envios_geo["lat_cliente"].between(-34, 6)
    & envios_geo["lon_cliente"].between(-74, -28)
)

# Crear una copia limpia sin modificar los datos originales
envios_geo_limpio = envios_geo.copy()

envios_geo_limpio.loc[
    ~(vendedor_valido & cliente_valido),
    "distancia_km"
] = np.nan

print(
    "Distancias faltantes antes:",
    envios_geo["distancia_km"].isna().sum()
)

print(
    "Distancias faltantes después:",
    envios_geo_limpio["distancia_km"].isna().sum()
)

print(
    "Artículos conservados:",
    len(envios_geo_limpio)
)

envios_geo_limpio["distancia_km"].describe().round(2)

Distancias faltantes antes: 554
Distancias faltantes después: 558
Artículos conservados: 112650


count    112092.00
mean        596.64
std         588.88
min           0.00
25%         183.07
50%         431.71
75%         791.78
max        3710.41
Name: distancia_km, dtype: float64

### 19. Agregación de la distancia por pedido

In [36]:
distancias_pedido = (
    envios_geo_limpio
    .groupby("order_id")
    .agg(
        distancia_maxima_km=("distancia_km", "max")
    )
    .reset_index()
)

print("Pedidos con artículos:", len(distancias_pedido))

print(
    "Pedidos sin distancia:",
    distancias_pedido["distancia_maxima_km"].isna().sum()
)

print(
    "Pedidos duplicados:",
    distancias_pedido["order_id"].duplicated().sum()
)

distancias_pedido["distancia_maxima_km"].describe().round(2)

Pedidos con artículos: 98666
Pedidos sin distancia: 493
Pedidos duplicados: 0


count    98173.00
mean       602.62
std        594.15
min          0.00
25%        188.21
50%        435.37
75%        800.68
max       3710.41
Name: distancia_maxima_km, dtype: float64

## Parte IV. Versión experimental V3

Se añade la distancia máxima disponible entre cada cliente y los vendedores del pedido. Se conservan los pedidos sin distancia para su posterior imputación dentro de los pipelines.

### 20. Incorporación de la distancia a la base histórica

In [37]:
train_v3 = train_v2.merge(
    distancias_pedido,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Dimensiones V2:", train_v2.shape)
print("Dimensiones V3:", train_v3.shape)

print(
    "Distancias faltantes en V3:",
    train_v3["distancia_maxima_km"].isna().sum()
)

print(
    "Pedidos duplicados:",
    train_v3["order_id"].duplicated().sum()
)

train_v3[
    ["order_id", "dias_prometidos", "distancia_maxima_km", "retraso"]
].head()

Dimensiones V2: (75128, 12)
Dimensiones V3: (75128, 13)
Distancias faltantes en V3: 390
Pedidos duplicados: 0


,order_id,dias_prometidos,distancia_maxima_km,retraso
0,bfbd0f9bdef84302105ad712db648a6c,19,566.040211,1
1,3b697a20d9e427646d92567910af6d57,24,708.535291,0
2,be5bc2f0da14d8071e2d45451ad119d9,35,915.734331,0
3,a41c8759fbe7aab36ea07e038b2d4465,57,818.048765,0
4,d207cc272675637bfed0062edffd0818,51,212.149919,0


### 21. Exportación de la versión V3

In [38]:
train_v3.to_csv(
    salida / "train_v3.csv",
    index=False
)

print(
    "Archivo guardado:",
    (salida / "train_v3.csv").exists()
)

print("Dimensiones:", train_v3.shape)

print(
    "Distancias faltantes:",
    train_v3["distancia_maxima_km"].isna().sum()
)

Archivo guardado: True
Dimensiones: (75128, 13)
Distancias faltantes: 390


## Conclusiones

1. Se integraron los datos de pedidos, clientes, artículos, productos y vendedores para construir una base inicial de **96.470 pedidos entregados**, con una etiqueta binaria de retraso y **ocho características** candidatas. Se identificaron **16 valores faltantes** en el peso total.
2. La separación cronológica produjo inicialmente **77.176 pedidos de entrenamiento** y **19.294 de prueba**. Se excluyeron **2.048** registros del entrenamiento cuyas fechas de entrega no permitían conocer la etiqueta antes del inicio de la prueba; quedaron **75.128 pedidos históricos**.
3. La tasa de retrasos del entrenamiento corregido fue del **7,11 %**, mientras que en prueba fue del **3,49 %**. Esta diferencia confirma que es necesario tener en cuenta el período de evaluación.
4. Se creó **V2**, que añade `dias_prometidos` a las ocho características iniciales. En los **96.470** pedidos estudiados, esta característica no presentó valores faltantes. El archivo `train_v2.csv` conserva los **75.128** pedidos históricos y añade la nueva columna.
5. Para explorar distancias logísticas, se agruparon las coordenadas en **19.015 prefijos postales únicos** mediante la mediana de latitud y longitud. La cobertura fue del **99,72 %** de los clientes y del **99,77 %** de los vendedores.
6. Se calcularon distancias aproximadas mediante Haversine y se detectaron coordenadas inconsistentes. Al marcar cuatro distancias extremas como faltantes, se conservaron los **112.650 artículos** sin perder registros. Posteriormente se construyó `distancia_maxima_km`, con una fila por pedido.
7. Se creó **V3** añadiendo `distancia_maxima_km` a V2. La base histórica resultante conserva **75.128 pedidos**, **10 características predictoras** y **390 distancias faltantes**. Los archivos `train.csv`, `train_v2.csv` y `train_v3.csv` se guardaron por separado para preservar los experimentos.
8. El preprocesamiento se definió sin ajustarlo a los datos de prueba; la imputación y otras transformaciones deben aprenderse exclusivamente dentro de los pipelines y de cada período de entrenamiento del cuadernillo 04.

**Limitaciones:** se analizan únicamente pedidos con entrega registrada, por lo que los resultados no representan necesariamente todos los pedidos. Las coordenadas representan prefijos postales y las distancias son aproximaciones en línea recta. El uso de `dias_prometidos` en una predicción al comprar depende de que la fecha estimada ya estuviera disponible. También debe respetarse la fecha en que se conoció cada etiqueta durante la validación temporal. Las comparaciones de desempeño entre V1, V2 y V3 se documentan en el cuadernillo 04.
